# Lesson 17 — LLM Fine-Tuning in Production

**Задача:** навчити Llama 3.1 8B витягувати структурований JSON для CRM із support email.

**План експерименту:**
1. Підготувати `eval.jsonl` і `train.jsonl` з матеріалів курсу.
2. Оцінити baseline Llama 3.1 8B без task fine-tuning.
3. Виконати fine-tuning через Unsloth + QLoRA.
4. Повторно оцінити fine-tuned модель на тих самих 30 eval-прикладах.
5. Порівняти метрики baseline vs fine-tuned.

> Notebook чистовий: невдала проміжна спроба з `PicklingError` прибрана.


## 1. Setup: GPU, packages, course data

Цей блок готує Colab, завантажує файли з репозиторію курсу та перевіряє кількість прикладів.


In [ ]:
!nvidia-smi

!pip install -q unsloth datasets accelerate bitsandbytes transformers trl

!rm -rf ai-engineering-course
!git clone --depth 1 https://github.com/andbilous/ai-engineering-course.git

!mkdir -p data results

!cp "ai-engineering-course/lesson 17 - llm-fine-tuning-in-production/homework/data/eval.jsonl" data/eval.jsonl
!cp "ai-engineering-course/lesson 17 - llm-fine-tuning-in-production/homework/data/train.jsonl" data/train.jsonl

!wc -l data/eval.jsonl data/train.jsonl

30 data/eval.jsonl
300 data/train.jsonl
330 total


## 2. Evaluation helper functions

Функції нижче використовуються і для baseline, і для fine-tuned моделі, щоб порівняння було чесним.


In [ ]:
import json
import torch
from pathlib import Path
from collections import defaultdict

FIELDS = ["customer_name", "product", "issue_category", "urgency", "summary"]

SYSTEM_PROMPT = (
    "You extract structured data from customer support emails. "
    "Return only a single valid JSON object with fields: "
    "customer_name (string or null), product (string), "
    "issue_category (one of: billing, technical, account, feature_request, other), "
    "urgency (one of: low, medium, high, critical), "
    "summary (one short sentence). No extra text."
)


def safe_json_parse(text: str):
    """
    Пробуємо витягнути JSON навіть якщо модель додала зайвий текст.
    Для ДЗ це важливо, бо json_valid — одна з основних метрик.
    """
    text = (text or "").strip()

    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text
        text = text.rsplit("```", 1)[0].strip()

    if text.startswith("json"):
        text = text[4:].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end > start:
        text = text[start:end + 1]

    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        return None, False


def field_match(predicted, expected, field):
    """
    Перевірка 5 полів:
    - category / urgency / product — майже exact match
    - customer_name — допускаємо null або ім'я всередині рядка
    - summary — м'яке порівняння за ключовими словами
    """
    if field == "summary":
        if not isinstance(predicted, str) or not isinstance(expected, str):
            return False

        predicted_words = {
            word.lower().strip(".,!?")
            for word in predicted.split()
            if len(word) > 3
        }

        expected_words = {
            word.lower().strip(".,!?")
            for word in expected.split()
            if len(word) > 3
        }

        if not predicted_words or not expected_words:
            return False

        overlap = len(predicted_words & expected_words) / max(len(expected_words), 1)
        return overlap >= 0.4

    if field == "customer_name":
        if predicted is None and expected is None:
            return True

        if predicted is None or expected is None:
            return False

        return expected.split()[0].lower() in str(predicted).lower()

    if predicted is None:
        return False

    return str(predicted).strip().lower() == str(expected).strip().lower()


def load_eval_examples(path="data/eval.jsonl"):
    examples = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                examples.append(json.loads(line))

    return examples


examples = load_eval_examples("data/eval.jsonl")

print(f"Loaded eval examples: {len(examples)}")
print("First eval example:")
print(json.dumps(examples[0], ensure_ascii=False, indent=2))

Loaded eval examples: 30
First eval example:
{
  "email": "Hi, can you tell me when the next maintenance window for Pro Plan is? — A.",
  "expected": {
    "customer_name": null,
    "product": "Pro Plan",
    "issue_category": "other",
    "urgency": "low",
    "summary": "Maintenance window schedule inquiry"
  }
}


## 3. Baseline Llama 3.1 8B

Baseline — це Llama 3.1 8B Instruct без fine-tuning на нашому task dataset.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import torch

BASE_MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

print("Loading baseline model:")
print(BASE_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16,
)

baseline_pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
)

print("Baseline model loaded successfully")

Loading baseline model:
unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
Baseline model loaded successfully


### 3.1. Sanity check на одному email

Перевіряємо, чи модель взагалі повертає валідний JSON.


In [ ]:
test_example = examples[0]

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_example["email"]},
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

output = baseline_pipe(
    prompt,
    max_new_tokens=180,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
)

raw_output = output[0]["generated_text"].strip()
parsed_json, is_valid = safe_json_parse(raw_output)

print("=" * 60)
print("EMAIL")
print("=" * 60)
print(test_example["email"])

print("\n" + "=" * 60)
print("EXPECTED")
print("=" * 60)
print(json.dumps(test_example["expected"], ensure_ascii=False, indent=2))

print("\n" + "=" * 60)
print("RAW MODEL OUTPUT")
print("=" * 60)
print(raw_output)

print("\n" + "=" * 60)
print("PARSED JSON")
print("=" * 60)
print("Valid JSON:", is_valid)
print(json.dumps(parsed_json, ensure_ascii=False, indent=2) if parsed_json else None)

EMAIL
Hi, can you tell me when the next maintenance window for Pro Plan is? — A.

EXPECTED
{
  "customer_name": null,
  "product": "Pro Plan",
  "issue_category": "other",
  "urgency": "low",
  "summary": "Maintenance window schedule inquiry"
}

RAW MODEL OUTPUT
{
  "customer_name": null,
  "product": "Pro Plan",
  "issue_category": "other",
  "urgency": "low",
  "summary": "Request for maintenance window information"
}

PARSED JSON
Valid JSON: True
{
  "customer_name": null,
  "product": "Pro Plan",
  "issue_category": "other",
  "urgency": "low",
  "summary": "Request for maintenance window information"
}


### 3.2. Full baseline evaluation

Оцінюємо baseline на всіх 30 eval-прикладах.


In [ ]:
import time
from collections import defaultdict

def generate_baseline_prediction(email: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": email},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    output = baseline_pipe(
        prompt,
        max_new_tokens=180,
        do_sample=False,
        return_full_text=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    return output[0]["generated_text"].strip()


baseline_results = []
json_valid_count = 0
exact_match_count = 0
field_correct = defaultdict(int)

total_input_tokens = 0
total_output_tokens = 0

start_time = time.time()

for index, example in enumerate(examples, start=1):
    email = example["email"]
    expected = example["expected"]

    raw_output = generate_baseline_prediction(email)
    predicted, valid_json = safe_json_parse(raw_output)

    if valid_json:
        json_valid_count += 1

    per_field_match = {}
    all_fields_match = True

    for field in FIELDS:
        predicted_value = predicted.get(field) if predicted else None
        expected_value = expected[field]

        is_match = valid_json and field_match(predicted_value, expected_value, field)
        per_field_match[field] = is_match

        if is_match:
            field_correct[field] += 1
        else:
            all_fields_match = False

    if valid_json and all_fields_match:
        exact_match_count += 1

    input_tokens = len(tokenizer.encode(email))
    output_tokens = len(tokenizer.encode(raw_output))

    total_input_tokens += input_tokens
    total_output_tokens += output_tokens

    baseline_results.append({
        "index": index,
        "email": email,
        "expected": expected,
        "raw_output": raw_output,
        "predicted": predicted,
        "valid_json": valid_json,
        "exact_match": valid_json and all_fields_match,
        "field_match": per_field_match,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    })

    status = "OK" if valid_json and all_fields_match else "CHECK"
    print(
        f"[{index:02d}/{len(examples)}] "
        f"{status} | valid_json={valid_json} | exact={valid_json and all_fields_match}"
    )

end_time = time.time()
elapsed_seconds = round(end_time - start_time, 2)

n = len(examples)

baseline_metrics = {
    "model": BASE_MODEL_NAME,
    "mode": "baseline_no_task_finetuning",
    "n_examples": n,
    "json_valid_rate": round(json_valid_count / n, 4),
    "exact_match_rate": round(exact_match_count / n, 4),
    "field_accuracy": {
        field: round(field_correct[field] / n, 4)
        for field in FIELDS
    },
    "total_input_tokens": total_input_tokens,
    "total_output_tokens": total_output_tokens,
    "avg_input_tokens": round(total_input_tokens / n, 1),
    "avg_output_tokens": round(total_output_tokens / n, 1),
    "elapsed_seconds": elapsed_seconds,
    "avg_seconds_per_example": round(elapsed_seconds / n, 2),
    "details": baseline_results,
}

Path("results").mkdir(exist_ok=True)

with open("results/baseline_8b_metrics.json", "w", encoding="utf-8") as file:
    json.dump(baseline_metrics, file, ensure_ascii=False, indent=2)

print()
print("=" * 60)
print("BASELINE RESULTS")
print("=" * 60)
print(f"Model: {baseline_metrics['model']}")
print(f"Examples: {baseline_metrics['n_examples']}")
print(f"JSON valid: {baseline_metrics['json_valid_rate'] * 100:.1f}%")
print(f"Exact match: {baseline_metrics['exact_match_rate'] * 100:.1f}%")
print("Field accuracy:")

for field in FIELDS:
    print(f"  {field:20s} {baseline_metrics['field_accuracy'][field] * 100:.1f}%")

print(f"Total input tokens: {baseline_metrics['total_input_tokens']}")
print(f"Total output tokens: {baseline_metrics['total_output_tokens']}")
print(f"Avg input tokens: {baseline_metrics['avg_input_tokens']}")
print(f"Avg output tokens: {baseline_metrics['avg_output_tokens']}")
print(f"Elapsed seconds: {baseline_metrics['elapsed_seconds']}")
print(f"Avg seconds/example: {baseline_metrics['avg_seconds_per_example']}")
print()
print("Saved to: results/baseline_8b_metrics.json")

BASELINE RESULTS
Model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
Examples: 30
JSON valid: 96.7%
Exact match: 30.0%
Field accuracy:
  customer_name        96.7%
  product              80.0%
  issue_category       76.7%
  urgency              53.3%
  summary              50.0%
Total input tokens: 724
Total output tokens: 1530
Avg input tokens: 24.1
Avg output tokens: 51.0
Elapsed seconds: 196.87
Avg seconds/example: 6.56

Saved to: results/baseline_8b_metrics.json


## 4. Free GPU memory after baseline

Після baseline модель видаляється з памʼяті GPU, щоб підготувати T4 до QLoRA fine-tuning.


In [ ]:
import gc
import torch

del baseline_pipe
del model
del tokenizer

gc.collect()
torch.cuda.empty_cache()

!nvidia-smi

GPU memory cleared. Tesla T4 is ready for fine-tuning.


## 5. Fine-tuning with Unsloth + QLoRA

> Практична примітка: у Colab після baseline я перезапустив runtime і запустив setup знову, щоб Unsloth імпортувався першим і не було проблем з памʼяттю.

Параметри fine-tuning:
- Base model: Llama 3.1 8B Instruct
- Method: QLoRA
- 4-bit loading: `True`
- LoRA rank: `r=16`
- LoRA alpha: `32`
- Epochs: `3`
- Train examples: `300`


### 5.1. Setup after runtime restart

Повторно підтягуємо дані після restart runtime.


In [ ]:
!nvidia-smi

!pip install -q unsloth datasets accelerate bitsandbytes transformers trl

!rm -rf ai-engineering-course
!git clone --depth 1 https://github.com/andbilous/ai-engineering-course.git

!mkdir -p data results

!cp "ai-engineering-course/lesson 17 - llm-fine-tuning-in-production/homework/data/eval.jsonl" data/eval.jsonl
!cp "ai-engineering-course/lesson 17 - llm-fine-tuning-in-production/homework/data/train.jsonl" data/train.jsonl

!wc -l data/eval.jsonl data/train.jsonl

30 data/eval.jsonl
300 data/train.jsonl
330 total


### 5.2. Load model, attach LoRA adapter, prepare dataset


In [ ]:
import unsloth

from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import json
from datasets import load_dataset

max_seq_length = 2048
dtype = None
load_in_4bit = True

FT_MODEL_NAME = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

# 1. Завантажуємо train dataset
train_dataset = load_dataset(
    "json",
    data_files="data/train.jsonl",
    split="train",
)

print("Train examples:", len(train_dataset))

# 2. Завантажуємо Llama 3.1 8B через Unsloth у 4-bit
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=FT_MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# 3. Додаємо LoRA adapter
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

# 4. Перетворюємо messages у text для SFT training
def format_messages_for_training(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

formatted_train_dataset = train_dataset.map(
    format_messages_for_training,
    remove_columns=["messages"],
)

print("Formatted examples:", len(formatted_train_dataset))
print("First formatted text preview:")
print(formatted_train_dataset[0]["text"][:1000])
print("Model + LoRA + dataset prepared.")

Train examples: 300
Formatted examples: 300
Model + LoRA + dataset prepared.


### 5.3. Train for 3 epochs

Checkpoint saving during training is disabled; final LoRA adapter is saved after training.


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import time
import json
from pathlib import Path

Path("results").mkdir(exist_ok=True)

training_args = TrainingArguments(
    output_dir="results/llama31_8b_lora_json_extractor",

    # Batch settings для T4
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # Вимога ДЗ: 3 epochs
    num_train_epochs=3,

    # Типові параметри для QLoRA fine-tuning
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",

    # FP16 для T4
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),

    # Логи
    logging_steps=5,

    # ВАЖЛИВО:
    # Не зберігаємо checkpoint після кожної епохи,
    # бо саме тут попередній запуск впав з PicklingError.
    save_strategy="no",
    save_only_model=True,

    optim="adamw_8bit",
    seed=3407,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=training_args,
)

start_time = time.time()

trainer_stats = trainer.train()

end_time = time.time()
training_seconds = round(end_time - start_time, 2)

ADAPTER_DIR = "results/llama31_8b_lora_adapter"

# Зберігаємо фінальний LoRA adapter
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

training_summary = {
    "model": FT_MODEL_NAME,
    "method": "QLoRA",
    "base_model": "Llama 3.1 8B Instruct",
    "load_in_4bit": True,
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": 3,
    "train_examples": len(formatted_train_dataset),
    "training_seconds": training_seconds,
    "training_minutes": round(training_seconds / 60, 2),
    "final_train_loss": float(trainer_stats.training_loss),
    "adapter_dir": ADAPTER_DIR,
}

with open("results/training_summary.json", "w", encoding="utf-8") as file:
    json.dump(training_summary, file, ensure_ascii=False, indent=2)

print("=" * 60)
print("TRAINING FINISHED")
print("=" * 60)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))
print("Saved adapter to:", ADAPTER_DIR)
print("Saved summary to: results/training_summary.json")

TRAINING FINISHED
{
  "model": "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
  "method": "QLoRA",
  "base_model": "Llama 3.1 8B Instruct",
  "load_in_4bit": true,
  "lora_r": 16,
  "lora_alpha": 32,
  "epochs": 3,
  "train_examples": 300,
  "training_seconds": 431.97,
  "training_minutes": 7.2,
  "final_train_loss": 0.24517662038928584,
  "adapter_dir": "results/llama31_8b_lora_adapter"
}
Saved adapter to: results/llama31_8b_lora_adapter
Saved summary to: results/training_summary.json


## 6. Fine-tuned evaluation

Оцінюємо fine-tuned модель на тих самих 30 eval-прикладах.


In [ ]:
import json
import torch
import time
from pathlib import Path
from collections import defaultdict
from unsloth import FastLanguageModel

Path("results").mkdir(exist_ok=True)

FIELDS = ["customer_name", "product", "issue_category", "urgency", "summary"]

SYSTEM_PROMPT = (
    "You extract structured data from customer support emails. "
    "Return only a single valid JSON object with fields: "
    "customer_name (string or null), product (string), "
    "issue_category (one of: billing, technical, account, feature_request, other), "
    "urgency (one of: low, medium, high, critical), "
    "summary (one short sentence). No extra text."
)


def safe_json_parse(text: str):
    text = (text or "").strip()

    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text
        text = text.rsplit("```", 1)[0].strip()

    if text.startswith("json"):
        text = text[4:].strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end > start:
        text = text[start:end + 1]

    try:
        return json.loads(text), True
    except json.JSONDecodeError:
        return None, False


def field_match(predicted, expected, field):
    if field == "summary":
        if not isinstance(predicted, str) or not isinstance(expected, str):
            return False

        predicted_words = {
            word.lower().strip(".,!?")
            for word in predicted.split()
            if len(word) > 3
        }

        expected_words = {
            word.lower().strip(".,!?")
            for word in expected.split()
            if len(word) > 3
        }

        if not predicted_words or not expected_words:
            return False

        overlap = len(predicted_words & expected_words) / max(len(expected_words), 1)
        return overlap >= 0.4

    if field == "customer_name":
        if predicted is None and expected is None:
            return True

        if predicted is None or expected is None:
            return False

        return expected.split()[0].lower() in str(predicted).lower()

    if predicted is None:
        return False

    return str(predicted).strip().lower() == str(expected).strip().lower()


def load_eval_examples(path="data/eval.jsonl"):
    examples = []

    with open(path, "r", encoding="utf-8") as file:
        for line in file:
            if line.strip():
                examples.append(json.loads(line))

    return examples


def generate_finetuned_prediction(email: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": email},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=max_seq_length,
    ).to("cuda")

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    output_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return output_text.strip()


# Перемикаємо модель у inference mode
FastLanguageModel.for_inference(model)

examples = load_eval_examples("data/eval.jsonl")

finetuned_results = []
json_valid_count = 0
exact_match_count = 0
field_correct = defaultdict(int)

total_input_tokens = 0
total_output_tokens = 0

start_time = time.time()

for index, example in enumerate(examples, start=1):
    email = example["email"]
    expected = example["expected"]

    raw_output = generate_finetuned_prediction(email)
    predicted, valid_json = safe_json_parse(raw_output)

    if valid_json:
        json_valid_count += 1

    per_field_match = {}
    all_fields_match = True

    for field in FIELDS:
        predicted_value = predicted.get(field) if predicted else None
        expected_value = expected[field]

        is_match = valid_json and field_match(predicted_value, expected_value, field)
        per_field_match[field] = is_match

        if is_match:
            field_correct[field] += 1
        else:
            all_fields_match = False

    if valid_json and all_fields_match:
        exact_match_count += 1

    input_tokens = len(tokenizer.encode(email))
    output_tokens = len(tokenizer.encode(raw_output))

    total_input_tokens += input_tokens
    total_output_tokens += output_tokens

    finetuned_results.append({
        "index": index,
        "email": email,
        "expected": expected,
        "raw_output": raw_output,
        "predicted": predicted,
        "valid_json": valid_json,
        "exact_match": valid_json and all_fields_match,
        "field_match": per_field_match,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
    })

    status = "OK" if valid_json and all_fields_match else "CHECK"
    print(
        f"[{index:02d}/{len(examples)}] "
        f"{status} | valid_json={valid_json} | exact={valid_json and all_fields_match}"
    )

end_time = time.time()
elapsed_seconds = round(end_time - start_time, 2)

n = len(examples)

finetuned_metrics = {
    "model": FT_MODEL_NAME,
    "mode": "finetuned_qlora_lora_adapter",
    "adapter_dir": "results/llama31_8b_lora_adapter",
    "n_examples": n,
    "json_valid_rate": round(json_valid_count / n, 4),
    "exact_match_rate": round(exact_match_count / n, 4),
    "field_accuracy": {
        field: round(field_correct[field] / n, 4)
        for field in FIELDS
    },
    "total_input_tokens": total_input_tokens,
    "total_output_tokens": total_output_tokens,
    "avg_input_tokens": round(total_input_tokens / n, 1),
    "avg_output_tokens": round(total_output_tokens / n, 1),
    "elapsed_seconds": elapsed_seconds,
    "avg_seconds_per_example": round(elapsed_seconds / n, 2),
    "details": finetuned_results,
}

with open("results/finetuned_8b_metrics.json", "w", encoding="utf-8") as file:
    json.dump(finetuned_metrics, file, ensure_ascii=False, indent=2)

print()
print("=" * 60)
print("FINETUNED RESULTS")
print("=" * 60)
print(f"Model: {finetuned_metrics['model']}")
print(f"Examples: {finetuned_metrics['n_examples']}")
print(f"JSON valid: {finetuned_metrics['json_valid_rate'] * 100:.1f}%")
print(f"Exact match: {finetuned_metrics['exact_match_rate'] * 100:.1f}%")
print("Field accuracy:")

for field in FIELDS:
    print(f"  {field:20s} {finetuned_metrics['field_accuracy'][field] * 100:.1f}%")

print(f"Total input tokens: {finetuned_metrics['total_input_tokens']}")
print(f"Total output tokens: {finetuned_metrics['total_output_tokens']}")
print(f"Avg input tokens: {finetuned_metrics['avg_input_tokens']}")
print(f"Avg output tokens: {finetuned_metrics['avg_output_tokens']}")
print(f"Elapsed seconds: {finetuned_metrics['elapsed_seconds']}")
print(f"Avg seconds/example: {finetuned_metrics['avg_seconds_per_example']}")
print()
print("Saved to: results/finetuned_8b_metrics.json")

FINETUNED RESULTS
Model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
Examples: 30
JSON valid: 100.0%
Exact match: 70.0%
Field accuracy:
  customer_name        93.3%
  product              96.7%
  issue_category       90.0%
  urgency              93.3%
  summary              83.3%
Total input tokens: 724
Total output tokens: 1260
Avg input tokens: 24.1
Avg output tokens: 42.0
Elapsed seconds: 93.07
Avg seconds/example: 3.1

Saved to: results/finetuned_8b_metrics.json


## 7. Results comparison

| Metric | Baseline 8B | Fine-tuned 8B | Change |
|---|---:|---:|---:|
| JSON valid | 96.7% | 100.0% | +3.3 п.п. |
| Exact match | 30.0% | 70.0% | +40.0 п.п. |
| customer_name | 96.7% | 93.3% | -3.4 п.п. |
| product | 80.0% | 96.7% | +16.7 п.п. |
| issue_category | 76.7% | 90.0% | +13.3 п.п. |
| urgency | 53.3% | 93.3% | +40.0 п.п. |
| summary | 50.0% | 83.3% | +33.3 п.п. |
| Avg output tokens | 51.0 | 42.0 | -9.0 tokens |
| Avg seconds/example | 6.56 sec | 3.10 sec | faster |

**Висновок:** QLoRA fine-tuning суттєво покращив якість на вузькій задачі extraction support email → CRM JSON. Найважливіше бізнес-покращення — `urgency`: 53.3% → 93.3%, що напряму впливає на правильну ескалацію тикетів.


## 8. Save artifacts

Зберігаємо метрики, training summary і LoRA adapter.


In [ ]:
!zip -r results/llama31_8b_lora_adapter.zip results/llama31_8b_lora_adapter

from google.colab import files

files.download("results/finetuned_8b_metrics.json")
files.download("results/training_summary.json")
files.download("results/llama31_8b_lora_adapter.zip")

Artifacts prepared for download:
- results/finetuned_8b_metrics.json
- results/training_summary.json
- results/llama31_8b_lora_adapter.zip
